# Random Forest Benchmarking

In this notebook, we compare our custom-built Random Forest classifier (using Bagging / Bootstrap Aggregation on our custom Decision Trees) against the `scikit-learn` implementation. We will evaluate its performance and execution time on the Breast Cancer dataset.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
from time import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as SklearnRF

# Import our custom modules
from classical_ml.ensemble.random_forest import RandomForest as CustomRF
from utils.metrics import accuracy_score, precision_score, recall_score, f1_score

d:\Github\Classical-ML-From-Scratch\classical_ml\tree_based\decision_tree.py:76: SyntaxWarning: invalid escape sequence '\s'
  Formula: Gain(S, A) = Entropy(S) - \sum (|Sv| / |S|) * Entropy(Sv)
d:\Github\Classical-ML-From-Scratch\classical_ml\tree_based\decision_tree.py:104: SyntaxWarning: invalid escape sequence '\s'
  Formula: -\sum p * log2(p)


In [2]:
# 1. Load Dataset
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (455, 30)
Testing data shape: (114, 30)


In [3]:
print("1. Custom Random Forest")
start = time()

# Initialize Random Forest with 10 trees
custom_rf = CustomRF(n_trees=10, max_depth=10)
custom_rf.fit(X_train, y_train)
preds_custom = custom_rf.predict(X_test)
time_custom = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_custom):.4f}")
print(f"Precision : {precision_score(y_test, preds_custom):.4f}")
print(f"Recall    : {recall_score(y_test, preds_custom):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_custom):.4f}")
print(f"Time Taken: {time_custom:.5f} seconds\n")

1. Custom Random Forest
Accuracy  : 0.9561
Precision : 0.9583
Recall    : 0.9718
F1-Score  : 0.9650
Time Taken: 17.74910 seconds



In [4]:
print("2. Scikit-Learn Random Forest")
start = time()

# We set n_estimators=10 to match our custom model (Sklearn default is 100)
# We use criterion='entropy' to match our custom Decision Tree logic
sk_rf = SklearnRF(n_estimators=10, criterion='entropy', max_depth=10, random_state=42)
sk_rf.fit(X_train, y_train)
preds_sk = sk_rf.predict(X_test)
time_sk = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_sk):.4f}")
print(f"Precision : {precision_score(y_test, preds_sk):.4f}")
print(f"Recall    : {recall_score(y_test, preds_sk):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_sk):.4f}")
print(f"Time Taken: {time_sk:.5f} seconds\n")

2. Scikit-Learn Random Forest
Accuracy  : 0.9649
Precision : 0.9589
Recall    : 0.9859
F1-Score  : 0.9722
Time Taken: 0.03327 seconds



## Conclusion
The custom Random Forest implementation successfully leverages **Bagging (Bootstrap Aggregation)** by training multiple independent Decision Trees on random subsets of the data and aggregating their predictions via majority voting.

As expected, the accuracy of the Random Forest is generally more stable compared to a single Decision Tree. The training time for the custom model is noticeably higher because it iteratively trains `n_trees` using pure Python `for-loops` and mathematically calculates the optimal splits using Entropy. `scikit-learn` remains exceptionally fast due to its highly optimized Cython backend and its ability to parallelize tree building using multiple CPU cores (`n_jobs`).